# Create Knowledge Assistant Brick

This notebook demonstrates how to create a Knowledge Assistant brick using the AgentBricksManager class.
The Knowledge Assistant will use the financial documents uploaded to our Unity Catalog volume.

In [ ]:
# Import configuration and helper functions (%run is unreliable on serverless)
import sys
sys.path.append('..')
exec(open('../config.py').read())
from resources.brick_setup_functions import AgentBricksManager

import logging
from databricks.sdk import WorkspaceClient
from databricks.sdk.runtime import dbutils

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Pre-flight check: verify schema and volume exist before proceeding
print("🔍 Pre-flight checks...")
try:
    volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
    dbutils.fs.ls(volume_path)
    print(f"  ✅ Volume exists: {volume_path}")
except Exception:
    raise Exception(
        f"❌ Volume not found: /Volumes/{catalog}/{schema}/{volume_name}\n"
        f"Please run 00_tables_setup.ipynb first to create the schema, volume, and upload data."
    )

In [0]:
# Initialize Workspace Client and AgentBricksManager
w = WorkspaceClient()
brick_manager = AgentBricksManager(w)

print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")
print(f"Using volume: {volume_name}")

In [ ]:
# Define the volume path for our uploaded documents
volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
print(f"Volume path: {volume_path}")

# Define knowledge sources - one per subdirectory in /data.
#
# The corpus has two halves that must stay distinguishable:
#
#   The original Mag7 folders (10Q, 10k, earning_releases, annual_reports,
#   earning_transcripts) hold documents committed to the repo.
#
#   The folders below them are built by the corpus scripts:
#     edgar_10k / edgar_10q  - real filings downloaded from SEC EDGAR
#                              (resources/fetch_filings.py)
#     synthetic              - GENERATED documents for organisations that publish
#                              no filing (resources/generate_synthetic_docs.py)
#
# The synthetic source gets its own knowledge source, and its description says
# plainly what it is. That description is visible to the assistant, so a question
# about Providence or Addepar retrieves material the model has been told is
# illustrative - on top of the watermark carried on every page of those documents.
# Do not merge the synthetic folder into the real-filing sources to tidy the list.

knowledge_sources = [
    {
        "files_source": {
            "name": "quarterly_reports_10q",
            "type": "files",
            "description": "10-Q quarterly financial reports from Apple, Meta, Amazon, Google, NVIDIA, Microsoft, and Tesla",
            "files": {"path": f"{volume_path}/10Q"}
        }
    },
    {
        "files_source": {
            "name": "annual_reports_10k",
            "type": "files",
            "description": "10-K annual financial reports and comprehensive business overviews",
            "files": {"path": f"{volume_path}/10k"}
        }
    },
    {
        "files_source": {
            "name": "earnings_releases",
            "type": "files",
            "description": "Quarterly earnings releases, press releases, and financial announcements",
            "files": {"path": f"{volume_path}/earning_releases"}
        }
    },
    {
        "files_source": {
            "name": "annual_reports",
            "type": "files",
            "description": "Annual reports and comprehensive business reviews",
            "files": {"path": f"{volume_path}/annual_reports"}
        }
    },
    {
        "files_source": {
            "name": "earnings_transcripts",
            "type": "files",
            "description": "Earnings call transcripts and Q&A sessions with management",
            "files": {"path": f"{volume_path}/earning_transcripts"}
        }
    },
    {
        "files_source": {
            "name": "sec_annual_filings",
            "type": "files",
            "description": (
                "Real annual filings (10-K, and 20-F for foreign issuers) downloaded from SEC EDGAR, "
                "covering automotive (Ford, Toyota, Honda Motor, American Honda Finance), pharmaceuticals "
                "and biotech (Pfizer, Merck, Amgen, Regeneron), chemicals (Dow), industrials (Deere, "
                "Rockwell Automation), airlines (United, Southwest), logistics (J.B. Hunt), financial "
                "information (S&P Global), insurance (Slide, Ryan Specialty) and digital health (Hinge "
                "Health). These are authoritative primary sources and may be cited as evidence."
            ),
            "files": {"path": f"{volume_path}/edgar_10k"}
        }
    },
    {
        "files_source": {
            "name": "sec_quarterly_filings",
            "type": "files",
            "description": (
                "Real 10-Q quarterly filings downloaded from SEC EDGAR for the same registrants, giving "
                "the most recent quarter of reported results. Authoritative primary sources."
            ),
            "files": {"path": f"{volume_path}/edgar_10q"}
        }
    },
    {
        "files_source": {
            "name": "synthetic_reference_documents",
            "type": "files",
            "description": (
                "GENERATED REFERENCE DOCUMENTS - NOT REAL FILINGS. Illustrative annual business reviews "
                "for organisations that publish no downloadable filing: FordDirect and Addepar (private "
                "companies), UCLA Health, Providence and Summa Health (nonprofit health systems that file "
                "no securities reports), Nationwide (a mutual insurer that suspended SEC reporting in "
                "2010), and Henkel and Munich Re (German issuers that report outside the SEC). Business "
                "descriptions reflect how these organisations are publicly known to operate, but every "
                "financial figure in them is invented for demonstration purposes. Treat these as "
                "illustrative context only. Never present a figure from these documents as a fact about "
                "the real organisation, and say clearly that the source is a generated document whenever "
                "you use one."
            ),
            "files": {"path": f"{volume_path}/synthetic"}
        }
    },
]

print(f"Configured {len(knowledge_sources)} knowledge source(s):")
for i, source in enumerate(knowledge_sources):
    source_info = source['files_source']
    print(f"  {i+1}. {source_info['name']}")
    print(f"      Path: {source_info['files']['path']}")

In [ ]:
# Create the Knowledge Assistant
ka_name = "Financial_Analysis_Assistant"
ka_description = "A Knowledge Assistant for analyzing financial documents and providing insights on company performance, earnings, and financial metrics."
ka_instructions = """You are a financial analysis assistant. Use the provided financial documents to answer questions about:
- Company financial performance and trends
- Earnings analysis and key metrics
- Risk factors and business outlook
- Quarterly and annual report insights

Always cite specific documents and sections when providing analysis. Be precise with numbers and dates.

The corpus contains two kinds of document and you must not blur them together.

Most documents are real SEC filings - 10-K, 10-Q and 20-F - and may be cited as
authoritative evidence.

A minority are GENERATED reference documents, written for this demonstration because
the organisation publishes no filing that could be obtained. They cover FordDirect,
Addepar, UCLA Health, Providence, Summa Health, Nationwide, Henkel and Munich Re. You
can recognise them three ways: the filename contains SYNTHETIC, the page header of
every page reads "SYNTHETIC DOCUMENT - NOT A REAL FILING", and the cover page says so.
Their business descriptions reflect how these organisations really operate, but every
financial figure in them is invented.

When you use one of these documents, say so in your answer and attribute the figure to
the generated document rather than to the organisation. Never state an invented figure
as a fact about the real organisation, and never combine figures from a generated
document with figures from a real filing in the same comparison without saying which is
which. If a question needs real reported financials for one of those organisations, say
that no such filing is available in this corpus and explain where the real disclosure
would be found."""

print(f"Creating Knowledge Assistant: {ka_name}")
print(f"Description: {ka_description}")
print("\nInstructions:")
print(ka_instructions)

In [0]:
# Create or update the Knowledge Assistant using the AgentBricksManager
import time
from requests.exceptions import ReadTimeout

try:
    result = brick_manager.ka_create_or_update(
        name=ka_name,
        knowledge_sources=knowledge_sources,
        description=ka_description,
        instructions=ka_instructions
    )
    
    print("\n✅ Knowledge Assistant created/updated successfully!")
    
    # Extract key information from the result
    tile_id = result['knowledge_assistant']['tile']['tile_id']
    operation = result.get('operation', 'unknown')
    
    print(f"Operation: {operation}")
    print(f"Tile ID: {tile_id}")
    print(f"Name: {result['knowledge_assistant']['tile']['name']}")
    
    # Check status
    status = brick_manager.ka_get_endpoint_status(tile_id)
    print(f"Status: {status}")
    
    # Display knowledge sources
    sources = result.get('knowledge_assistant', {}).get('knowledge_sources', [])
    print(f"\nKnowledge Sources ({len(sources)}):")
    for i, source in enumerate(sources):
        print(f"  {i+1}. {source.get('name', 'Unknown')} - {source.get('description', 'No description')}")

except ReadTimeout:
    # The operation timed out but may still be processing in the background
    print("⏱️ Request timed out after 20 seconds, but the operation may still be processing...")
    print("\nThis typically happens when indexing large documents.")
    print("Checking if Knowledge Assistant exists...\n")
    
    # Try to find the KA by name
    try:
        ka_result = brick_manager.find_by_name(ka_name)
        if ka_result:
            tile_id = ka_result.tile_id
            print(f"✅ Found Knowledge Assistant: {ka_name}")
            print(f"Tile ID: {tile_id}")
            
            # Try to check status (this is a quick call)
            try:
                status = brick_manager.ka_get_endpoint_status(tile_id)
                print(f"Current Status: {status}")
                
                if status == 'PROVISIONING':
                    print("\n🔄 The Knowledge Assistant is still being set up.")
                    print("This can take several minutes for large document sets.")
                    print("Run the next cell to wait for completion.")
                elif status == 'ONLINE':
                    print("\n✅ Knowledge Assistant is ready to use!")
            except:
                print("Status check timed out - the KA is likely still provisioning.")
                print("Run the next cell to wait for completion.")
        else:
            print("❌ Could not find the Knowledge Assistant.")
            print("The creation may have failed. Try running this cell again.")
    except Exception as find_error:
        print(f"❌ Error checking for existing KA: {find_error}")
        raise

except Exception as e:
    error_msg = str(e)
    
    # Check if the error is ALREADY_EXISTS
    if 'ALREADY_EXISTS' in error_msg:
        print("ℹ️ Knowledge Assistant already exists. Retrieving existing one...")
        
        # Find the existing KA by name
        ka_result = brick_manager.find_by_name(ka_name)
        
        if ka_result:
            tile_id = ka_result.tile_id
            print(f"✅ Found existing Knowledge Assistant: {ka_name}")
            print(f"Tile ID: {tile_id}")
            
            # Try to get status (quick call)
            try:
                status = brick_manager.ka_get_endpoint_status(tile_id)
                print(f"Status: {status}")
            except:
                print("Status: (unable to retrieve - may be provisioning)")
            
            print("\nℹ️ The Knowledge Assistant was already created in a previous run.")
            print("You can proceed to the next cells to check status and add examples.")
        else:
            print("❌ Could not find the existing Knowledge Assistant.")
            raise
    else:
        print(f"❌ Error creating Knowledge Assistant: {e}")
        raise

In [0]:
# Wait for the Knowledge Assistant to be ready (if it's still provisioning)
print("\n🔄 Checking Knowledge Assistant status...")

current_status = brick_manager.ka_get_endpoint_status(tile_id)
print(f"Current status: {current_status}")

if current_status in ['PROVISIONING', 'NOT_READY']:
    print("Knowledge Assistant is still provisioning. You can check its status in the Databricks workspace.")
    print(f"Navigate to Machine Learning > Knowledge Assistants to view the assistant with ID: {tile_id}")
elif current_status == 'ONLINE':
    print("✅ Knowledge Assistant is ready to use!")
    print(f"You can test it in the Databricks workspace or use the tile_id: {tile_id}")
else:
    print(f"Knowledge Assistant status: {current_status}")

In [0]:
# Add some example questions to help users understand what the KA can do
# First check if the Knowledge Assistant is ready

current_status = brick_manager.ka_get_endpoint_status(tile_id)
print(f"Knowledge Assistant Status: {current_status}")

if current_status != 'ONLINE':
    print(f"\n⏳ Knowledge Assistant is not ready yet (status: {current_status})")
    print("Cannot add examples until the status is ONLINE.")
    print("\nWait for document indexing to complete, then re-run this cell.")
    print("This typically takes 5-15 minutes for large document sets.")
else:
    example_questions = [
        {
            "question": "What were the key revenue trends in the most recent quarterly report?",
            "guideline": "Focus on revenue growth, segment performance, and year-over-year comparisons"
        },
        {
            "question": "What are the main risk factors mentioned in the 10-K filing?",
            "guideline": "Identify and summarize the primary business and financial risks disclosed"
        },
        {
            "question": "How did earnings per share change compared to the previous quarter?",
            "guideline": "Provide specific EPS figures and calculate percentage changes"
        },
        {
            "question": "What guidance did management provide for the upcoming quarter or year?",
            "guideline": "Extract forward-looking statements and management guidance from earnings calls or reports"
        }
    ]

    print(f"\n📚 Adding {len(example_questions)} example questions...")

    try:
        # Add examples in batch for efficiency
        created_examples = brick_manager.ka_add_examples_batch(tile_id, example_questions)
        print(f"✅ Successfully added {len(created_examples)} example questions")
        
        for i, example in enumerate(created_examples):
            question_text = example.get('question', 'Unknown question')
            print(f"  {i+1}. {question_text[:60]}{'...' if len(question_text) > 60 else ''}")
            
    except Exception as e:
        print(f"⚠️ Warning: Could not add example questions: {e}")
        print("The Knowledge Assistant was created successfully, but examples failed to add.")

In [0]:
# Summary and next steps
print("\n🎉 Knowledge Assistant Setup Complete!")
print("\n📋 Summary:")
print(f"  • Name: {ka_name}")
print(f"  • Tile ID: {tile_id}")
print(f"  • Status: {brick_manager.ka_get_endpoint_status(tile_id)}")
print(f"  • Knowledge Sources: {len(knowledge_sources)} configured")
print(f"  • Volume Path: {volume_path}")

print("\n🚀 Next Steps:")
print("1. Wait for the Knowledge Assistant to be ONLINE if it's still provisioning")
print("2. Test the assistant in the Databricks workspace (Machine Learning > Knowledge Assistants)")
print("3. Try asking questions about financial performance, earnings, and company metrics")
print("4. Use the example questions provided to get started")

print(f"\n🔗 Access your Knowledge Assistant:")
print(f"   Navigate to Machine Learning > Knowledge Assistants in your Databricks workspace")
print(f"   Look for: {ka_name}")